# Demonstration

The following cell blocks are for exploring the data in the nemo (wigits workflow) parquet output directory, as produced by [nemo-operator](../nemo-operator).

Set up the Python environment and run the notebook like so:

```
conda create -n oncoglue python=3.13
conda activate oncoglue

pip install notebook
cd nemo-query-examples
jupyter notebook
```

In [ ]:
# install the dependencies
! pip install duckdb pyarrow pandas matplotlib

In [ ]:
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

In [ ]:
import os

NEMO_IMAGE_TAG = "0.1.1"
BASE = f"../nemo-operator/data/output/nemo/{NEMO_IMAGE_TAG}"
# first run dir found; set explicitly to pick another
RUN_ID = sorted(os.listdir(BASE))[0]
# expose to `!` shell cells
os.environ["BASE"], os.environ["RUN_ID"] = BASE, RUN_ID
print(BASE, RUN_ID)

In [ ]:
! ls "$BASE"

In [ ]:
! ls "$BASE/$RUN_ID/"

In [ ]:
tbl_metadata = f"{BASE}/*/metadata.parquet"
tbl_purple_qc = f"{BASE}/*/*_purple_qc.parquet"
tbl_alignments_dupfreq = f"{BASE}/*/*_alignments_dupfreq.parquet"
tbl_neo_predictions = f"{BASE}/*/*_neo_predictions.parquet"

In [ ]:
# https://duckdb.org/docs/stable/data/multiple_files/overview
duckdb.sql(f'select distinct input_id, input_prefix from read_parquet("{BASE}/*/*_purple_qc.parquet") limit 10')

In [ ]:
duckdb.sql(f'select input_id, pkg_versions from read_parquet("{tbl_metadata}")')

In [ ]:
# check parquet compression
duckdb.sql(f'select distinct row_group_id, compression from parquet_metadata("{BASE}/*/metadata.parquet")')

In [ ]:
r = duckdb.read_parquet(tbl_alignments_dupfreq)

In [ ]:
df = duckdb.sql('select * from r limit 10')

In [ ]:
print(df)

In [ ]:
duckdb.sql(f'select input_id, output_id, version as purple_version from read_parquet("{BASE}/*/*_purple_version.parquet") group by input_id, output_id, version order by version')

In [ ]:
duckdb.sql(f'select input_id, output_id, version as linx_version from read_parquet("{BASE}/*/*_linx_version.parquet") group by input_id, output_id, version order by version')

In [ ]:
duckdb.sql(f'''
    select input_id, input_prefix, ne_id, variant_type, variant_info, gene_name, peptide_count
    from read_parquet("{tbl_neo_predictions}")
    where peptide_count > 50
''')

In [ ]:
df1 = duckdb.sql(f'select * from read_parquet("{tbl_neo_predictions}")').to_df()
x_col = "gene_name"
y_col = "peptide_count"
df1.plot(x=x_col, y=y_col, kind="line", figsize=(10, 4), title=f"{y_col} by {x_col}")
plt.tight_layout()
plt.show()

In [ ]:
duckdb.sql(f'select * from parquet_scan("{tbl_alignments_dupfreq}")')

In [ ]:
duckdb.sql(f'''
    select
        input_id,
        input_prefix,
        output_id,
        avg(cast(duplicate_read_count as double)) as average_duplicate_read_count,
        avg(frequency)
    from
        parquet_scan("{tbl_alignments_dupfreq}")
    group by input_id, input_prefix, output_id
''')

In [ ]:
df = duckdb.sql(f'select * from parquet_scan("{tbl_alignments_dupfreq}")')

In [ ]:
df.describe()

In [ ]:
duckdb.sql(f'describe "{tbl_alignments_dupfreq}"')

In [ ]:
# print arrow schema using pyarrow
schema = pq.read_schema(f"{BASE}/{RUN_ID}/metadata.parquet")
print(schema)

In [ ]:
# create temporary table query technique
duckdb.sql('drop table if exists tmp_alignments_dupfreq')

# note `... with no data` statement, this causes reading the table structure only
duckdb.sql(f'create table tmp_alignments_dupfreq as select * from "{tbl_alignments_dupfreq}" with no data')

In [ ]:
duckdb.sql('describe tmp_alignments_dupfreq')

In [ ]:
# should output zero rows as no data - structure only
duckdb.sql('select * from tmp_alignments_dupfreq')

In [ ]:
arrow_tbl = pq.read_table(f"{BASE}/{RUN_ID}/metadata.parquet")
metadata = arrow_tbl.to_pydict()
assert len(metadata['files']) == 1 # `files` attribute is in nested list of list of dicts
files = metadata['files'][0]

tables = set()

for file in files:
    tables.add(file['tbl'])
print(len(tables))
print("-" * 32)

for tbl in sorted(tables):
    print(tbl)

In [ ]:
# print out the schema of each table
for tbl in sorted(tables):
    print(tbl)
    duckdb.sql(f'describe "{BASE}/*/*_{tbl}.parquet"').show(max_rows=1000)